# 04 — Đánh giá, so sánh, chọn model cuối

Metric hồi quy: **MAE** (USD dễ giải thích), **RMSE** (phạt đuôi), **R²**. Không dùng MAPE vì charges gần 1k vẫn ổn nhưng MAE/RMSE đủ.

Cách đọc:

1. So với Dummy: mọi model thật đều hơn hẳn (Dummy R² ≈ 0).
2. Train vs test: DT gần khớp; RF khe nhỏ (0.894 → 0.880); không overfit nặng.
3. Lỗi thực tế: dự đoán thấp một ca bệnh nặng (underestimate charges) có thể làm công ty lỗ — RMSE/MAE đều quan trọng.
4. Deploy: RF ~37 ms, ~0.5 MB — chấp nhận được.


In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))
from evaluate import main
main()

meta = json.loads(Path("../models/metadata.json").read_text())
print("CHOSEN", meta["model_display_name"], meta["metrics"])
print("reason:", meta["chosen_reason"])
print("sklearn", meta["library_versions"])


## Vì sao Random Forest, không phải SVR?

SVR có **MAE thấp nhất** (1847 USD) nhưng RMSE cao hơn RF → SVR ổn trên đa số mẫu, kém trên đuôi charges cao (đúng với EDA: nhóm smoker + BMI cao). RF giải thích nhiều phương sai hơn (R² 0.8795) và RMSE thấp nhất — phù hợp bài toán phí bảo hiểm nơi sai số lớn ở ca đắt là đáng lo.

Linear Regression kém vì không bắt tương tác smoker × bmi (hình 4). Decision Tree đã tốt sau khi hãm `max_depth=4`, nhưng một cây đơn vẫn kém ổn định hơn rừng.
